# BanglaT5 fine-tune — seed 42 (v2, clean notebook)

Fresh notebook. The previous one accumulated stale editor state that kept
reintroducing a fixed bug, so this replaces it entirely.

## ⚠️ TWO SETTINGS BEFORE YOU RUN

1. **Settings → Accelerator → `GPU T4 x2`**
2. **Settings → Internet → On**

Then **Save Version → Save & Run All**.

**P100 will not work.** It is sm_60 and Kaggle's PyTorch ships no sm_60 kernels.
Cell 1 hard-fails on it in seconds rather than dying 6 minutes into setup.

**Do not run interactively** — interactive sessions die when the browser
disconnects and cannot be tracked via the API. This is a ~6 hour run.

## Config
| | |
|---|---|
| GPU | single T4 (`CUDA_VISIBLE_DEVICES=0`; the second card is left idle) |
| Precision | fp32 — T4 has no bf16 hardware, and T5 diverges to NaN in fp16 |
| Optimizer | Adafactor (~2 GB less state than AdamW) |
| Effective batch | 8 × 8 = 64 |
| Epochs | 2 → 3,180 steps, ~4.6 s/it, **~5h40m** |
| Checkpoints | every 500 steps, `save_total_limit=2` |

Single GPU is deliberate: it removes distributed eval padding, rank contention on
the weight cache, and duplicated datasets in host RAM — four separate failures.

In [ ]:
# ══ 1 — HARDWARE GATE ═══════════════════════════════════════════════════════
import torch

n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4 x2"
names = [torch.cuda.get_device_name(i) for i in range(n)]
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__}\nGPUs: {n} — {names}\nsm_{cap[0]}{cap[1]}")

assert cap[0] >= 7, (
    f"\n\n*** WRONG ACCELERATOR: {names[0]} (sm_{cap[0]}{cap[1]}) ***\n"
    f"Kaggle's PyTorch supports sm_70+. P100 is sm_60 and CANNOT run this.\n"
    f"Fix: Settings -> Accelerator -> GPU T4 x2 -> Save Version -> Save & Run All\n"
)
print("\n✅ hardware OK")

In [ ]:
# ══ 2 — normalizer (REQUIRED by BanglaT5's model card, not optional) ════════
!pip install -q git+https://github.com/csebuetnlp/normalizer
from normalizer import normalize
print("normalizer OK:", normalize("হেলো,   নাসেনিয়া ডকে  আপনাকে স্বাগতম।"))

In [ ]:
# ══ 3 — locate inputs ═══════════════════════════════════════════════════════
import glob, os, shutil, sys

print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/02_train_t5.py", recursive=True)
assert hits, "Attach the code dataset: Add Input -> Datasets -> nascenia-code"
CODE = os.path.dirname(hits[0])

# competition mounts at /kaggle/input/competitions/<slug>/, not /kaggle/input/<slug>/
raw = glob.glob("/kaggle/input/**/train.csv", recursive=True)
assert raw, "Attach the competition: Add Input -> Competitions -> Nascenia AI Hackathon"
RAW = os.path.dirname(raw[0])

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")
print("CODE:", CODE, "\nRAW :", RAW, "\nfiles:", sorted(os.listdir("/kaggle/working/code")))

In [ ]:
# ══ 4 — frozen dev split ════════════════════════════════════════════════════
# seed 42 / dev-size 5000 governs the DATA SPLIT and must never change — it is what
# makes dev numbers comparable across runs, machines and accounts.
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

In [ ]:
# ══ 5 — warm the HF cache (files only, no model instantiated) ═══════════════
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from huggingface_hub import snapshot_download
print("cache warm:", snapshot_download("csebuetnlp/banglat5"))

In [ ]:
# ══ 6 — CONFIG CHECK ════════════════════════════════════════════════════════
# Confirms the attached code dataset has the current defaults BEFORE committing
# ~6 h of GPU. group_by_length must be False: enabling it without a precomputed
# `length` column causes a ~14-minute silent stall that looks exactly like a hang.
import subprocess

h = subprocess.run(["python", "02_train_t5.py", "--help"],
                   cwd="/kaggle/working/code", capture_output=True, text=True).stdout
assert "--group-by-length" in h, "code dataset is stale — re-attach latest nascenia-code"
assert "OFF by" in h, "stale code dataset: group_by_length is not default-off"
print("✅ code dataset current — group_by_length defaults OFF")

In [ ]:
# ══ 7 — SMOKE TEST (~3 min) ═════════════════════════════════════════════════
# Validates tokenization, normalizer, metric callback, generation, 3B assert and
# checkpoint saving. The SCORE it prints is meaningless (~30 steps against
# warmup_steps=1000 means the LR never leaves zero). Success = exit 0.
import subprocess, shlex, os

ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0",
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
       "TOKENIZERS_PARALLELISM": "false"}

cmd = ("python 02_train_t5.py "
       "--data-dir /kaggle/working/processed --out-dir /kaggle/working/runs "
       "--smoke --max-train 1000 --epochs 1 --batch-size 8 --grad-accum 2 "
       "--eval-subset 100 --eval-steps 20 --eval-batch-size 8 --gen-num-beams 2 "
       "--no-bertscore-eval --run-name smoke")

r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code", env=ENV)
assert r.returncode == 0, "SMOKE FAILED — do not start the full run"
print("\n✅ smoke passed")

In [ ]:
# ══ 8 — FULL RUN (~5h40m) ═══════════════════════════════════════════════════
# lr 3e-4 + warmup 200: the previous run used warmup 1000 of only 3,180
# total steps — 31% of training spent ramping LR from zero, which left it
# undertrained (token_f1 0.070 at epoch 0.63 vs 0.2669 for a constant string).
# Watch for: "device: Tesla T4  (n_gpu=1)" and "precision = fp32".
# First loss line appears at step 100 (~8 min) — silence before that is normal.
import subprocess, shlex, os, time

SEED = 42
cmd = (f"python 02_train_t5.py "
       f"--data-dir /kaggle/working/processed --out-dir /kaggle/working/runs "
       f"--model csebuetnlp/banglat5 --seed {SEED} "
       f"--epochs 2 --lr 3e-4 --warmup 200 --optim adafactor "
       f"--batch-size 8 --grad-accum 8 --eval-batch-size 8 "
       f"--max-source-len 384 --max-target-len 256 "
       f"--eval-subset 300 --eval-steps 500 "
       f"--gen-num-beams 4 --gen-min-new-tokens 80 "
       f"--precision auto --run-name banglat5_seed{SEED}")

print(cmd + "\n" + "=" * 70, flush=True)
t0 = time.time()
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code", env=ENV)
print(f"\nexit {r.returncode} after {(time.time()-t0)/60:.1f} min")
assert r.returncode == 0, "training failed — see traceback above"

In [ ]:
# ══ 9 — run record → copy into LOCAL_EXPERIMENTS.md ═════════════════════════
import json, glob
for f in sorted(glob.glob("/kaggle/working/runs/*/run.json")):
    print(f"\n=== {f} ===")
    print(json.dumps(json.load(open(f)), indent=2, ensure_ascii=False))

---
## After it finishes

1. Save `/kaggle/working/runs/banglat5_seed42/best` as a **Kaggle Dataset** so the decode notebook can attach it without retraining.
2. Copy `run.json` into `LOCAL_EXPERIMENTS.md`.
3. **Judge on Token F1 / ROUGE-L, not composite** — the local composite is mis-calibrated by ~0.118.

| Bar | Token F1 |
|---|---|
| LB constant (currently #1) | 0.2669 |
| Frequency-only constant | **0.3519** — below this the model beat nothing but unigram stats |

Also watch **`eval_pred_tokens`**: references average ~100 tokens. If it stays near
the 256 cap past epoch 1, the model isn't learning to stop and length needs capping
at decode time.